# NYC Taxi analytics pipeline - ComPar 26/27, Project 1

Ingestion and cleaning (M1), temporal demand (M2), zone statistics (M3)
and the optional operations (O1-O4). Every operation runs inside its own
`with profiler.profile(...)` block, and at the end all metrics (totals and
per-stage breakdown) are saved to `results/<RUN_NAME>.json`.
`analysis.ipynb` reads those files to build the report's tables and charts.

**Two ways to run it:**

* **Interactively:** edit the *Parameters* cell, then *Kernel -> Restart
  Kernel and Run All Cells*. Restarting matters, because `getOrCreate()`
  reuses an existing Spark session and ignores any new configuration.
* **From the command line** (required by the brief): papermill overrides
  the *Parameters* cell. The executed copy, with all outputs, is saved to
  `results/notebooks/`.

  ```bash
  scripts/run_pipeline.sh local   -p YEARS 2024 -p FRACTION 0.1 -p RUN_NAME local-smoke
  scripts/run_pipeline.sh cluster -p YEARS 2024 -p RUN_NAME cluster-3w-baseline
  ```

## Parameters

This cell is tagged `parameters`, so papermill can override any of these values with `-p NAME VALUE`.

In [1]:
MASTER = "local[*]"            # "spark://spark-master:7077" inside the Docker cluster
DATA_DIR = None                # None -> ../data locally, /data on the cluster
OUTPUT_DIR = None              # None -> ../output locally, /output on the cluster
RESULTS_DIR = None             # None -> ../results locally, /results on the cluster
YEARS = "2024"                 # "2024" | "2019-2024" | "2024,2026"
MONTHS = "1-12"                # "1-12" | "1-3"
FRACTION = None                # sample fraction e.g. 0.1, or None for every record
OPS = "m2,m3,o2,o3"            # operations to run after M1 (M1 always runs)
RUN_NAME = None                # label for results/<RUN_NAME>.json; None -> timestamp
EXPLAIN = False                # save the formatted physical plan of every operation
DRIVER_MEMORY = "8g"           # local mode: the driver is also the only executor
EXECUTOR_MEMORY = "3g"         # cluster mode: must fit within the workers' --memory
SPARK_CONF = {}                # non-default settings, e.g. {"spark.sql.shuffle.partitions": "64"}

# O4 classification thresholds (fraction of a bucket's trips)
AIRPORT_THRESHOLD = 0.3
NEGOTIATED_THRESHOLD = 0.1
GROUP_THRESHOLD = 0.2

In [2]:
import os
import time

on_cluster = MASTER.startswith("spark://")
DATA_DIR = DATA_DIR or ("/data" if on_cluster else "../data")
OUTPUT_DIR = OUTPUT_DIR or ("/output" if on_cluster else "../output")
RESULTS_DIR = RESULTS_DIR or ("/results" if on_cluster else "../results")
RUN_NAME = RUN_NAME or time.strftime("run-%Y%m%d-%H%M%S")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


def parse_range(spec, lo, hi):
    """'2024' -> [2024]; '2019-2021' -> [2019, 2020, 2021]; '2019,2023' -> [2019, 2023]."""
    values = []
    for part in str(spec).split(","):
        if "-" in part:
            a, b = part.split("-")
            values.extend(range(int(a), int(b) + 1))
        else:
            values.append(int(part))
    return sorted(v for v in set(values) if lo <= v <= hi)


years = parse_range(YEARS, 2009, 2026)
months = parse_range(MONTHS, 1, 12)
ops = [o.strip().lower() for o in str(OPS).split(",") if o.strip()]
print(f"run={RUN_NAME} years={years} months={months} fraction={FRACTION} ops={ops}")
print(f"data={DATA_DIR} output={OUTPUT_DIR} results={RESULTS_DIR}")

run=run-20261013-132541 years=[2024] months=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] fraction=None ops=['m2', 'm3', 'o2', 'o3']
data=../data output=../output results=../results


## Spark session

The baseline deliberately uses Spark defaults. Each experiment's
non-default settings go in `SPARK_CONF`, so they are recorded in the
results file.

**Java:** Spark 4.0 runs only on Java 17 or 21. Java 23+ fails with
`UnsupportedOperationException: getSubject is not supported`. Jupyter and
PyCharm kernels do not inherit a `JAVA_HOME` exported in your terminal, so
the next cell picks a supported JDK itself (local mode only). If the
session already failed once in this kernel, **restart the kernel**: the
JVM has already started with the wrong Java.

In [3]:
import platform
import subprocess

SUPPORTED_JAVA = (17, 21)


def java_major(java_home):
    """Major version of the JDK at `java_home`, read from its `release` file."""
    try:
        with open(os.path.join(java_home, "release")) as f:
            for line in f:
                if line.startswith("JAVA_VERSION="):
                    return int(line.split("=", 1)[1].strip().strip('"').split(".")[0])
    except (OSError, ValueError):
        pass
    return None


if not on_cluster and java_major(os.environ.get("JAVA_HOME", "")) not in SUPPORTED_JAVA:
    if platform.system() == "Darwin":
        for version in ("21", "17"):
            found = subprocess.run(["/usr/libexec/java_home", "-v", version],
                                   capture_output=True, text=True)
            if found.returncode == 0:
                os.environ["JAVA_HOME"] = found.stdout.strip()
                break
    if java_major(os.environ.get("JAVA_HOME", "")) not in SUPPORTED_JAVA:
        raise RuntimeError("Spark 4.0 needs Java 17 or 21: install one and set JAVA_HOME "
                           "before starting Jupyter")

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(cluster: Java from the Docker image)"))

JAVA_HOME: /Users/rloucao/Library/Java/JavaVirtualMachines/ms-21.0.8/Contents/Home


In [4]:
from pyspark.sql import SparkSession

builder = (SparkSession.builder
           .master(MASTER)
           .appName(f"taxi-{RUN_NAME}")
           .config("spark.driver.memory", DRIVER_MEMORY)
           .config("spark.ui.showConsoleProgress", "false")).config("spark.sql.session.timeZone", "UTC")


if on_cluster:
    builder = builder.config("spark.executor.memory", EXECUTOR_MEMORY)
for key, value in SPARK_CONF.items():
    builder = builder.config(key, value)

spark = builder.getOrCreate()
sc = spark.sparkContext

print("Spark version:", spark.version)
print("Spark UI:", sc.uiWebUrl)
print("Default parallelism (cores seen by the driver):", sc.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 13:25:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
Spark UI: http://unknown1aa448f27515.lan:4040
Default parallelism (cores seen by the driver): 10


In [5]:
import sys
sys.path.append(os.path.abspath("."))   # spark_profiler.py lives next to this notebook

from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)

## Ingestion

The monthly files are **not schema-compatible** across 2009-2026:

* column names changed (2009 uses `Trip_Pickup_DateTime`, `Fare_Amt`, ...;
  2010 uses `pickup_datetime`, ...);
* types changed too (`passenger_count` is a double in some years and
  int64 in others).

Reading everything with a single `spark.read.parquet(glob)` either fails
or silently produces nulls. Instead, each file is read on its own, mapped
onto the canonical schema below, and the results are unioned.

**Note:** the 2009-2010 files have pickup/dropoff lat/lon instead of
`PULocationID`/`DOLocationID`. Those columns come out null for these
years, so M1 removes their rows. Decide (and document in the report)
whether that is acceptable, or restrict the study to 2011+.

In [6]:
from functools import reduce

import pyspark.sql.functions as F
from pyspark.sql import DataFrame, Window
from pyspark.sql.types import (DoubleType, IntegerType, StringType, StructField,
                               StructType, TimestampType)

# Canonical (name, type) of every column the pipeline uses.
CANONICAL_COLUMNS = [
    ("tpep_pickup_datetime", TimestampType()),
    ("tpep_dropoff_datetime", TimestampType()),
    ("PULocationID", IntegerType()),
    ("DOLocationID", IntegerType()),
    ("passenger_count", IntegerType()),
    ("trip_distance", DoubleType()),
    ("RatecodeID", IntegerType()),
    ("payment_type", IntegerType()),
    ("fare_amount", DoubleType()),
    ("tip_amount", DoubleType()),
    ("total_amount", DoubleType()),
]

# Older spellings of the canonical columns (matched case-insensitively).
# TODO: verify against the 2009/2010 files; their payment_type is a string
# ("CASH", "Credit", ...) and needs mapping to the numeric codes.
COLUMN_ALIASES = {
    "tpep_pickup_datetime": ["Trip_Pickup_DateTime", "pickup_datetime"],
    "tpep_dropoff_datetime": ["Trip_Dropoff_DateTime", "dropoff_datetime"],
    "RatecodeID": ["Rate_Code", "rate_code"],
    "fare_amount": ["Fare_Amt"],
    "tip_amount": ["Tip_Amt"],
    "total_amount": ["Total_Amt"],
}

ZONES_SCHEMA = StructType([
    StructField("LocationID", IntegerType(), False),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

In [7]:
def month_files(data_dir, years, months):
    """(year, month, path) for every requested month that exists on disk."""
    found = []
    for y in years:
        for m in months:
            path = os.path.join(data_dir, f"yellow_tripdata_{y}-{m:02d}.parquet")
            if os.path.exists(path):
                found.append((y, m, path))
            else:
                print(f"[ingest] missing {path} - skipped")
    if not found:
        raise FileNotFoundError(f"no yellow_tripdata files for years={years} in {data_dir}")
    return found


def read_month(path, year, month):
    df = spark.read.parquet(path)
    by_lower = {c.lower(): c for c in df.columns}
    exprs = []
    for name, dtype in CANONICAL_COLUMNS:
        candidates = [name] + COLUMN_ALIASES.get(name, [])
        src = next((by_lower[c.lower()] for c in candidates if c.lower() in by_lower), None)
        col = F.col(src) if src is not None else F.lit(None)
        exprs.append(col.cast(dtype).alias(name))
    # The file's own year/month: pickup timestamps contain stray out-of-range
    # dates, so per-year statistics are keyed on the source file instead.
    return df.select(*exprs, F.lit(year).alias("source_year"), F.lit(month).alias("source_month"))


def load_trips(data_dir, years, months, fraction=None, seed=42):
    files = month_files(data_dir, years, months)
    print(f"[ingest] reading {len(files)} monthly files")
    trips = reduce(DataFrame.unionByName, (read_month(path, y, m) for y, m, path in files))
    if fraction is not None and fraction < 1.0:
        trips = trips.sample(fraction=fraction, seed=seed)
    return trips


def load_zones(path):
    return spark.read.option("header", "true").schema(ZONES_SCHEMA).csv(path)

In [8]:
zones = load_zones(os.path.join(DATA_DIR, "taxi_zone_lookup.csv"))
trips = load_trips(DATA_DIR, years, months, FRACTION)

print("trips partitions:", trips.rdd.getNumPartitions())
trips.printSchema()

[ingest] reading 12 monthly files
trips partitions: 120
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_year: integer (nullable = false)
 |-- source_month: integer (nullable = false)



## M1 - Data ingestion and cleaning

A record is removed if it violates **any** of these criteria. They are not
mutually exclusive, so the per-criterion counts can add up to more than
`removed`, which is reported separately.

In [9]:
LOCATION_MIN, LOCATION_MAX = 1, 263

# Removed the "passanger_count" column because it is not required
# Only O1 uses it, so instead of filtering it out, we can just ignore it in the criteria
# About 10% of the records have passenger_count=0, which is invalid, but we don't need to filter them out for m2/m3
REQUIRED = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID",
            "DOLocationID", "trip_distance", "fare_amount"]


def criteria():
    """name -> boolean Column that is True when the record is invalid."""
    pu, do, pc = F.col("PULocationID"), F.col("DOLocationID"), F.col("passenger_count")
    raw = {
        "null_required": reduce(lambda a, b: a | b, (F.col(c).isNull() for c in REQUIRED)),
        "fare_le_0": F.col("fare_amount") <= 0,
        "distance_le_0": F.col("trip_distance") <= 0,
        "bad_passenger_count": (pc == 0) | (pc > 6),
        "dropoff_not_after_pickup": F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"),
        "location_out_of_range": ~pu.between(LOCATION_MIN, LOCATION_MAX)
                                 | ~do.between(LOCATION_MIN, LOCATION_MAX),
    }
    # Comparisons against null yield null; nulls are covered by `null_required`.
    return {name: F.coalesce(cond, F.lit(False)) for name, cond in raw.items()}


def is_invalid():
    return reduce(lambda a, b: a | b, criteria().values())


def cleaning_stats(trips):
    """Per source year: total records, removed per criterion, total removed."""
    aggs = [F.count(F.lit(1)).alias("total")]
    aggs += [F.sum(F.when(cond, 1).otherwise(0)).alias(name) for name, cond in criteria().items()]
    aggs += [F.sum(F.when(is_invalid(), 1).otherwise(0)).alias("removed")]
    return trips.groupBy("source_year").agg(*aggs).orderBy("source_year")


def clean_trips(trips):
    return trips.filter(~is_invalid())

In [10]:
import pandas as pd

with profiler.profile("m1"):
    m1_rows = cleaning_stats(trips).collect()   # tiny: one row per year

m1_stats = pd.DataFrame([r.asDict() for r in m1_rows]).set_index("source_year")
m1_stats.to_csv(os.path.join(OUTPUT_DIR, "m1_cleaning_stats.csv"))

# Lazy: every operation below re-reads and re-filters the Parquet files.
# That is the naive baseline on purpose (see caching in the optimisation study).
clean = clean_trips(trips)
m1_stats

[m1] wall time: 3.12s (4 stages, 123 tasks) - see profiler.report() / profiler.compare(...)


,total,null_required,fare_le_0,distance_le_0,bad_passenger_count,dropoff_not_after_pickup,location_out_of_range,removed
source_year,,,,,,,,
2024,41169720,0,748284,776305,401638,13510,425660,2210145


## Shared helpers

`run_operation` builds an operation's (lazy) result outside the profiler,
then profiles only the write, which is the action that actually runs the
jobs. Operations that are not in `OPS`, or not implemented yet, are skipped.

In [11]:
import contextlib
import io


def with_time_features(df):
    """Derived columns used by several operations."""
    pickup, dropoff = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")
    return (df
            .withColumn("pickup_date", F.to_date(pickup))
            .withColumn("day_of_week", F.dayofweek(pickup))          # 1 = Sunday ... 7 = Saturday
            .withColumn("hour_of_day", F.hour(pickup))
            .withColumn("pickup_month", F.date_trunc("month", pickup))
            .withColumn("duration_min",
                        (F.unix_timestamp(dropoff) - F.unix_timestamp(pickup)) / 60.0))


def with_pickup_zone(df, zones):
    """Annotate trips with pickup Borough/Zone/service_zone (zone join #1)."""
    pu = zones.select(F.col("LocationID").alias("PULocationID"),
                      F.col("Borough").alias("pickup_borough"),
                      F.col("Zone").alias("pickup_zone"),
                      F.col("service_zone").alias("pickup_service_zone"))
    return df.join(pu, on="PULocationID", how="left")


def with_dropoff_zone(df, zones):
    """Annotate trips with dropoff Borough/Zone/service_zone (zone join #2)."""
    do = zones.select(F.col("LocationID").alias("DOLocationID"),
                      F.col("Borough").alias("dropoff_borough"),
                      F.col("Zone").alias("dropoff_zone"),
                      F.col("service_zone").alias("dropoff_service_zone"))
    return df.join(do, on="DOLocationID", how="left")


def run_operation(name, build, write):
    """Build `name`'s lazy result, then profile its write (the action)."""
    if name not in ops:
        print(f"[{name}] not in OPS - skipped")
        return None
    try:
        result = build()
    except NotImplementedError as e:
        print(f"[{name}] skipped: {e}")
        return None
    with profiler.profile(name):
        write(result)
    if EXPLAIN:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            result.explain(mode="formatted")
        with open(os.path.join(RESULTS_DIR, f"{RUN_NAME}-{name}-plan.txt"), "w") as f:
            f.write(buf.getvalue())
    return result

## M2 - Temporal demand analysis

For each `(pickup_borough, day_of_week, hour_of_day)` triple compute:

* total trip count
* mean `fare_amount`
* mean trip duration in minutes (derived from the timestamps)
* mean `trip_distance`

The output is written as Parquet, **partitioned by borough**.

*Performance angle:* zone join (broadcast vs. sort-merge), group-by shuffle, partitioned write (number of output files).

In [12]:
# (no actions here - only transformations)
def build_m2(clean: DataFrame, zones: DataFrame):
    trips = with_pickup_zone(with_time_features(clean), zones)
    return (trips
            .groupBy("pickup_borough", "day_of_week", "hour_of_day")
            .agg(F.count("*").alias("trip_count"),
                 F.avg("fare_amount").alias("avg_fare"),
                 F.avg("duration_min").alias("avg_duration_min"),
                 F.avg("trip_distance").alias("avg_distance")))
    

In [13]:
m2_result = run_operation("m2", lambda: build_m2(clean, zones),
                        write=lambda df: df.write.mode("overwrite").partitionBy("pickup_borough").parquet(f"{OUTPUT_DIR}/m2"))

[m2] wall time: 5.32s (3 stages, 123 tasks) - see profiler.report() / profiler.compare(...)


## M3 - Zone-level statistics

For each pickup zone and month: mean and standard deviation of
`fare_amount` and `trip_distance`. Then, using window functions:

* rank zones by monthly trip volume within each borough
* compute each zone's rolling 3-month trip share within its borough
* flag zones whose current-month volume deviates from their own rolling
  mean by more than 1.5 standard deviations

*Performance angle:* several windows with different partitionings, each adding an Exchange + Sort; window partition sizes.

In [14]:
def build_m3(clean: DataFrame, zones: DataFrame):
    # Part A: one row per (zone, month); "month" is the source file's month
    monthly = (with_pickup_zone(clean, zones)
               .withColumn("month", F.make_date("source_year", "source_month", F.lit(1)))
               .withColumn("month_idx", F.col("source_year") * 12 + F.col("source_month"))
               .groupBy("pickup_borough", "PULocationID", "pickup_zone", "month", "month_idx")
               .agg(F.count("*").alias("trip_count"),
                    F.avg("fare_amount").alias("avg_fare"),
                    F.stddev("fare_amount").alias("std_fare"),
                    F.avg("trip_distance").alias("avg_distance"),
                    F.stddev("trip_distance").alias("std_distance")))

    # Part B: windows (rangeBetween works on month numbers, so missing months are handled)
    by_borough_month = Window.partitionBy("pickup_borough", "month").orderBy(F.desc("trip_count"))
    zone_last_3m = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-2, 0)
    borough_last_3m = Window.partitionBy("pickup_borough").orderBy("month_idx").rangeBetween(-2, 0)
    zone_prev_3m = Window.partitionBy("PULocationID").orderBy("month_idx").rangeBetween(-3, -1)

    return (monthly
            .withColumn("volume_rank", F.rank().over(by_borough_month))
            .withColumn("rolling_3m_share",
                        F.sum("trip_count").over(zone_last_3m) / F.sum("trip_count").over(borough_last_3m))
            .withColumn("prev_3m_mean", F.avg("trip_count").over(zone_prev_3m))
            .withColumn("prev_3m_std", F.stddev("trip_count").over(zone_prev_3m))
            .withColumn("prev_3m_months", F.count("trip_count").over(zone_prev_3m))
            .withColumn("volume_anomaly",
                        (F.col("prev_3m_months") == 3)
                        & (F.abs(F.col("trip_count") - F.col("prev_3m_mean")) > 1.5 * F.col("prev_3m_std"))))

In [15]:
m3_result = run_operation("m3", lambda: build_m3(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/m3"))

[m3] wall time: 4.63s (7 stages, 126 tasks) - see profiler.report() / profiler.compare(...)


## O1 - Fare anomaly detection

For each `(PULocationID, hour_of_day)` bucket compute the mean and σ of
`fare_amount / trip_distance`. Flag trips whose per-mile fare exceeds
mean + 3σ, then compare weekday vs. weekend anomaly rates by
`payment_type` and pickup borough.

*Performance angle:* aggregate, then join the bucket statistics back to the full trip table.

In [ ]:
def build_o1(clean: DataFrame, zones: DataFrame):
    # TODO: implement - return a lazy DataFrame (no actions here)
    raise NotImplementedError("o1 not implemented yet")

In [ ]:
o1_result = run_operation("o1", lambda: build_o1(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o1"))

## O2 - Origin-destination flow analysis

* Monthly trip counts for all `(PULocationID, DOLocationID)` pairs, with
  the zone lookup joined to resolve zone names and boroughs.
* Each route's share of total outbound trips from its pickup zone per month.
* Top-20 routes per pickup borough, and routes whose monthly share
  reverses direction between Q1 and Q3.

*Performance angle:* large shuffle on a high-cardinality key, two zone joins, skew (Manhattan and airport routes).

In [ ]:
def build_o2(clean: DataFrame, zones: DataFrame):
    # TODO: implement - return a lazy DataFrame (no actions here)
    raise NotImplementedError("o2 not implemented yet")

In [ ]:
o2_result = run_operation("o2", lambda: build_o2(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o2"))

## O3 - Demand forecasting features

For each `(PULocationID, day_of_week, hour_of_day)` bucket: mean historical
trip count, variance, and a 4-week-lag value modelled as a **self-join on
offset pickup dates**. Demand index = current count ÷ historical mean;
flag buckets with an index above 2.0. Join the zone lookup to annotate
results by `service_zone`.

*Performance angle:* self-join (shuffles both sides), window partition sizing.

In [ ]:
def build_o3(clean: DataFrame, zones: DataFrame):
    # TODO: implement - return a lazy DataFrame (no actions here)
    raise NotImplementedError("o3 not implemented yet")

In [ ]:
o3_result = run_operation("o3", lambda: build_o3(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o3"))

## O4 - Trip-type classification

Per `(PULocationID, hour_of_day)` bucket: fraction of airport trips
(`RatecodeID` ∈ {2, 3}), negotiated fares (`RatecodeID` = 5) and group
trips (`passenger_count` ≥ 4). Classify each bucket as residential /
commercial / transit / mixed using `AIRPORT_THRESHOLD`,
`NEGOTIATED_THRESHOLD` and `GROUP_THRESHOLD`. Join the classification back
to the full trip table and compute fare and distance statistics per
zone type per month.

*Performance angle:* join-back of a small table to a huge one, Python UDF vs. built-in `when` chain.

In [ ]:
def build_o4(clean: DataFrame, zones: DataFrame):
    # TODO: implement - return a lazy DataFrame (no actions here)
    raise NotImplementedError("o4 not implemented yet")

In [ ]:
o4_result = run_operation("o4", lambda: build_o4(clean, zones),
                        write=lambda df: df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/o4"))

## Summary

Totals per operation for this run. `profiler.stages("<op>")` shows the
per-stage breakdown; `skew_max_over_median` ≥ 10 marks a skewed stage
(one task taking 10× the median).

In [ ]:
profiler.report()

In [ ]:
for name in profiler.runs:
    stages = profiler.stages(name)
    skewed = stages[stages["skew_max_over_median"].fillna(0) >= 10]
    print(f"{name}: {len(stages)} stages, {len(skewed)} skewed")
    if len(skewed):
        display(skewed)

## Save results

Writes `results/<RUN_NAME>.json`, which holds every operation's totals and
per-stage metrics plus the configuration needed to reproduce the run.

In [ ]:
import platform

executors = sc._jsc.sc().statusTracker().getExecutorInfos()
metadata = {
    "run_name": RUN_NAME,
    "master": sc.master,
    "spark_version": spark.version,
    "num_executors": max(len(executors) - 1, 1),   # minus the driver; local mode = 1
    "total_cores": sc.defaultParallelism,
    "years": years, "months": months, "fraction": FRACTION, "ops": ops,
    "data_dir": DATA_DIR, "host": platform.node(),
    "spark_conf": {k: v for k, v in sc.getConf().getAll()
                   if k.startswith(("spark.sql", "spark.executor", "spark.driver.memory",
                                    "spark.default"))
                   and not any(x in k for x in ("extraJavaOptions", "warehouse", "artifact", "executor.id"))},
    "total_wall_time_s": round(sum(r["wall_time_s"] for r in profiler.runs.values()), 2),
}
results_path = os.path.join(RESULTS_DIR, f"{RUN_NAME}.json")
profiler.save(results_path, metadata)
print("saved", results_path, "| total wall time:", metadata["total_wall_time_s"], "s")

In [ ]:
# Keep the session open while exploring interactively; stop it to release
# the cluster's executors (and always when running through papermill).
spark.stop()